In [1]:
import numpy as np
from pyscf import gto, scf, cc

####  test H2 monomers ####
a = 2 # bond length in a cluster
d = 4 # distance between each cluster
unit = 'b' # unit of length
na = 2 # size of a cluster (monomer)
nc = 1 # set as integer multiple of monomers
spin = 0 # spin per monomer
frozen = 0 # frozen orbital per monomer
elmt = 'H'
unit = 'B'
basis = 'ccpvdz'
atoms = ""
for n in range(nc*na):
    shift = ((n - n % na) // na) * (d-a)
    atoms += f"{elmt} {n*a+shift:.5f} 0.00000 0.00000 \n"
###########################

mol = gto.M(atom=atoms,
            basis=basis,
            verbose=4,
            unit=unit,
            symmetry=0,
            charge=0,
            spin=spin*nc,
            max_memory=40000,
            )

mf = scf.RHF(mol)
mf.kernel()

mycc = cc.CCSD(mf).set_frozen()
mycc.kernel()

System: uname_result(system='Linux', node='sharmagroup-rn', release='7.0.0-30-generic', version='#30~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Fri Aug  7 13:27:52 UTC 2', machine='x86_64')  Threads 16
Python 3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:20:58) [GCC 14.3.0]
numpy 2.4.4  scipy 1.17.1  h5py 3.16.0
Date: Mon Aug 31 21:33:31 2026
PySCF version 2.14.0
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT ORIG_HEAD 3d1768f5e33b144b606c3d2c81c12ee54d794501
GIT HEAD      c63a953ba603a5ad8c1d65d88da72aaf05ede4d8

[ENV] OLD_PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:
[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:/home/sharmagroup/sharmagroup/pyscf-forge:
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 2
[INPUT] num. electrons = 2
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry 0 subgroup None
[INPUT] Mole.unit = B
[INPUT] Symbol           X                Y                Z      unit          X 

(np.float64(-0.04140461073842018),
 array([[-8.09152987e-17,  1.51749956e-02, -2.39526610e-16,
          2.19896028e-17,  8.52137207e-17, -9.32720550e-03,
          4.43767189e-17, -2.67542551e-16,  1.71172289e-17]]),
 array([[[[-1.21556393e-01,  1.39748679e-16,  7.02098466e-02,
           -1.84451129e-18,  1.48304411e-17,  9.50283316e-17,
            2.69710962e-17,  1.95977969e-18,  8.06991662e-03],
          [ 1.39748679e-16, -4.70963368e-02,  7.38847847e-17,
            2.20094176e-18, -1.00568970e-16, -1.40103948e-02,
           -8.38108787e-17, -1.09128074e-17,  1.09412947e-17],
          [ 7.02098466e-02,  7.38847847e-17, -5.24900891e-02,
            1.35418531e-18, -3.43143218e-18, -3.17057313e-17,
           -2.48854415e-17, -1.25533567e-18, -8.97267756e-03],
          [-1.84451129e-18,  2.20094176e-18,  1.35418531e-18,
           -3.16461698e-02,  6.75974600e-18, -5.97200193e-18,
           -8.42328211e-18, -4.03069663e-17,  8.41052988e-19],
          [ 1.48304411e-17, -1.005

In [15]:
options = {'eql_time': 10,
            'n_blocks': 100,
            'n_walkers': 300,
            'max_error': 0.0,
            'mix_precision': False,
            'seed': 17,
            'walker_type': 'rhf',
            'trial': 'rpt2ccsd_bar',
            'free_projection': False,
            }

from afqmc import integral, launch_afqmc
integral.prep_integral(mycc, chol_cut=1e-5)


Preparing AFQMC calculation
CCSD type input object
Calculating Cholesky integrals
Cholesky shape: (46, 10, 10) 
Finished calculating Cholesky integrals
Size of the correlation space:
Number of electrons:        [1, 1]
Number of basis functions:  10
Number of Cholesky vectors: 46


E0831 22:19:03.058046  461286 cuda_executor.cc:1213] [0] Failed to track device allocation: ALREADY_EXISTS: Allocation at address (nil) (size 0) is already tracked


In [16]:
import time

import numpy as np

from afqmc import config, prep, sampling

from functools import partial

print = partial(print, flush=True)
init_time = time.time()

prep.print_start()
config.setup_jax()

ham_data, ham, prop, trial, wave_data, sampler, options = prep.init_afqmc(options)

if "rdm1" not in wave_data:
    wave_data["rdm1"] = trial.get_rdm1(wave_data)
ham_data = ham.build_measurement_intermediates(ham_data, trial, wave_data)
ham_data = ham.build_propagation_intermediates(ham_data, prop, trial, wave_data)
h0 = ham_data['h0']

prop_data = prep.init_hf_prop_data(trial, wave_data, ham_data, options)

init_e = prop_data["e_estimate"]
init_w = np.sum(prop_data["weights"])

print("\nEquilibration")

print(f"{'1/T':>5s}  "
      f"{'nodes':>5s}  {'weight':>12s}  "
      f"{'energy':>12s}  {'runTime':>8s}")
print(f"{0.:5.2f}  "
      f"{0:5d}  {init_w:12.5f}  "
      f"{init_e:12.5f}  {time.time() - init_time:8.2f}")

block_time = prop.dt * options["n_prop_steps"]
neql_block = int(-(-options["eql_time"] // block_time))

sampler_eq = sampling.sampler(
    n_prop_steps = options["n_prop_steps"],
    n_chol = sampler.n_chol,
    n_blocks = neql_block,
    )

for n in range(sampler_eq.n_blocks):
    prop_data, (wt, e, _ ) \
        = sampler_eq.block_sample(prop_data, ham_data, prop, trial, wave_data)
    nodes = prop_data["n_killed_walkers"]

    if (n+1) % (min(max(neql_block // 10, 1), 20)) == 0 and n > 0:
        print(f"{(n+1)*block_time:5.2f}  "
              f"{nodes:5d}  {wt:12.5f}  "
              f"{e:12.5f}  {time.time() - init_time:8.2f}")
        # prop_data = prop.stochastic_reconfiguration_local(prop_data)
        # prop_data["overlaps"] = trial.calc_overlap(prop_data["walkers"], wave_data)
        prop_data["n_killed_walkers"] = 0


    ________                     _____                    
    ___  __ \___  __________________(_)_____________ _    
    __  /_/ /  / / /_  __ \_  __ \_  /__  __ \_  __ `/    
    _  _, _// /_/ /_  / / /  / / /  / _  / / /  /_/ /     
    /_/ |_| \__,_/ /_/ /_//_/ /_//_/  /_/ /_/_\__, /      
                                             /____/       
    _____________________________  ___________            
    ___    |__  ____/_  __ \__   |/  /_  ____/            
    __  /| |_  /_   _  / / /_  /|_/ /_  /                 
    _  ___ |  __/   / /_/ /_  /  / / / /___               
    /_/  |_/_/      \___\_\/_/  /_/  \____/               

Hostname:     sharmagroup-rn
System:       Linux
Node:         sharmagroup-rn
Release:      7.0.0-30-generic
Machine:      x86_64
Processor:    x86_64
JAX backend:  GPU
JAX devices:  [CudaDevice(id=0)]
Device kind:  NVIDIA GeForce RTX 5060 Ti
Platform:     gpu

QMC Parameters
eql_time        -         10
n_blocks        -        100
n_walkers     

In [17]:
print("\nSampling")
print(f"Target (raw) 0.6 x max_error = {0.75 * options['max_error']:.5f}")
print(f"{'blocks':>6s}  {'nodes':>5s}  "
      f"{'weight':>10s}  {'E_Guide':>12s}  {'error':>8s}  "
      f"{'weightp':>10s}  {'E_Trial':>12s}  {'error':>8s}  "
      f"{'Walltime':>10s}")

wt_sp = np.zeros(sampler.n_blocks, dtype="float64")
eg_sp = np.zeros(sampler.n_blocks, dtype="float64")
wp_sp = np.zeros(sampler.n_blocks, dtype="complex128")
t2_sp = np.zeros(sampler.n_blocks, dtype="complex128")
e0_sp = np.zeros(sampler.n_blocks, dtype="complex128")
e1_sp = np.zeros(sampler.n_blocks, dtype="complex128")

nodes = 0
for n in range(sampler.n_blocks):
    prop_data, (wt, eg, wp, t2, e0, e1) =\
        sampler.block_sample(prop_data, ham_data, prop, trial, wave_data)
    
    wt_sp[n] = wt
    eg_sp[n] = eg
    wp_sp[n] = wp
    t2_sp[n] = t2
    e0_sp[n] = e0
    e1_sp[n] = e1
    
    nodes += prop_data["n_killed_walkers"]
    prop_data["n_killed_walkers"] = 0

    if (n+1) % (min(max(sampler.n_blocks // 10, 1), 20)) == 0 and n > 0:
        
        weight, eguide, eg_err = sampling.blocking(wt_sp[:n+1], eg_sp[:n+1], final=False)

        weighp, ept2, ept2_err \
            = sampling.pt2blocking(h0, wp_sp[:n+1], t2_sp[:n+1], e0_sp[:n+1], e1_sp[:n+1], final=False)
        
        print(f"{n+1:6d}  {nodes:5d}  "
              f"{weight:10.5f}  {eguide:12.5f}  {eg_err:8.5f}  "
              f"{weighp.real:10.5f}  {ept2:12.5f}  {ept2_err:8.5f}  "
              f"{time.time() - init_time:10.2f}")
        
        prop_data["e_estimate"] = 0.8 * prop_data["e_estimate"] + 0.2 * eguide.real
        
        if ept2_err < 0.75 * options["max_error"] and n > 120:
            break


Sampling
Target (raw) 0.6 x max_error = 0.00000
blocks  nodes      weight       E_Guide     error     weightp       E_Trial     error    Walltime
    10      0   299.81390      -1.13002   0.00087   300.04787      -1.13071   0.00003       11.03
    20      0   299.79027      -1.12993   0.00063   300.02364      -1.13072   0.00002       11.17
    30      0   299.78117      -1.12986   0.00054   300.01333      -1.13072   0.00002       11.31
    40      0   299.78113      -1.12975   0.00044   300.01604      -1.13072   0.00001       11.45
    50      0   299.77725      -1.13001   0.00042   300.00785      -1.13071   0.00001       11.59
    60      0   299.76631      -1.12941   0.00043   299.99938      -1.13073   0.00001       11.73
    70      0   299.76364      -1.12893   0.00041   299.99144      -1.13075   0.00001       11.87
    80      0   299.77477      -1.12907   0.00038   300.00585      -1.13074   0.00001       12.01
    90      0   299.77224      -1.12918   0.00035   300.00404      -1

In [18]:
ot, t2, e0, e1 = trial._calc_energy_pt(prop_data["walkers"][0], ham_data, wave_data)
print(ot, t2, e0, e1)
print(h0 + e0 + e1 - t2 * e0)

(0.34092337683469254+0.82581932066458j) (0.001750750720068371+0.0008547193507892491j) (-1.5954140643422852+0.002197836776427118j) (-0.03812831466762483-0.003591560688972141j)
(-1.1307473281543328-3.394050354927074e-05j)


In [19]:
trial

pt2ccsd_bar(norb=10, nelec=(1, 1), n_batch=1, nchol_chunk=46, mix_precision=False)

In [56]:
import jax
jax.config.update("jax_enable_x64", True)
from jax import jit, lax, random
from jax import numpy as jnp
import opt_einsum as oe

# NOTE: these use ham_data["chol_bar"] / ham_data["h1_bar"] / wave_data["exp_t1"],
# which are only built for the *bar* trial.  Set options['trial'] = 'rpt2ccsd_bar'
# above (currently 'rpt2ccsd') or these cells will KeyError on "chol_bar".


def _prop_chol(self, walker, ham_data, wave_data, floor=1.0e-6, uniform_mix=0.01):
    """Sampling probability for each Cholesky vector.

    Scores every vector by |its contribution to the two-body energy| evaluated at
    `walker` -- pass the HF/initial determinant to get a fixed reference guide --
    and then

        pi_g = (1 - u) * |e2_g| / sum_g |e2_g|  +  u / nchol

    `floor` zeroes guided weight that is negligible relative to the largest score,
    so numerical noise cannot drive the proposal.  The uniform component then keeps
    pi_g >= u / nchol > 0 for *every* vector.  That positivity is not cosmetic: it
    is what bounds the importance weights 1/pi_g and keeps the estimator unbiased.
    Never zero a probability outright -- a vector with pi_g = 0 is never sampled,
    yet still contributes to the energy, which would bias the result.
    """
    nocc, norb = self.nelec[0], self.norb

    chol = ham_data["chol_bar"]
    walker_bar = wave_data['exp_t1'] @ walker

    green = (walker_bar.dot(jnp.linalg.inv(walker_bar[: walker_bar.shape[1], :]))).T

    gl = oe.contract("ir,gqr->giq", green, chol, backend="jax")
    gl_c = oe.contract("gii->g", gl[:, :, :nocc], backend="jax")
    e2_0_c_g = oe.contract("g,g->g", gl_c, gl_c, backend="jax") * 2
    e2_0_e_g = -oe.contract("gij,gji->g", gl[:, :, :nocc], gl[:, :, :nocc], backend="jax")

    e2_g = jnp.abs(e2_0_c_g + e2_0_e_g)

    # floor guard: drop guided weight below `floor` x the largest score
    e2_g = jnp.where(e2_g >= floor * jnp.max(e2_g), e2_g, 0.0)

    # uniform guard: mix so that every vector keeps a strictly positive probability
    nchol = e2_g.shape[0]
    uniform = jnp.full((nchol,), 1.0 / nchol)
    total = jnp.sum(e2_g)
    guided = jnp.where(total > 0.0, e2_g / jnp.where(total > 0.0, total, 1.0), uniform)
    pi_g = (1.0 - uniform_mix) * guided + uniform_mix * uniform

    return pi_g


def _prop_chol_in_place(self, e2_g_estimate, floor=1.0e-6, uniform_mix=0.01):
    """Sampling probability for each Cholesky vector.

    Scores every vector by |its contribution to the two-body energy| evaluated at
    `walker` -- pass the HF/initial determinant to get a fixed reference guide --
    and then

        pi_g = (1 - u) * |e2_g| / sum_g |e2_g|  +  u / nchol

    `floor` zeroes guided weight that is negligible relative to the largest score,
    so numerical noise cannot drive the proposal.  The uniform component then keeps
    pi_g >= u / nchol > 0 for *every* vector.  That positivity is not cosmetic: it
    is what bounds the importance weights 1/pi_g and keeps the estimator unbiased.
    Never zero a probability outright -- a vector with pi_g = 0 is never sampled,
    yet still contributes to the energy, which would bias the result.
    """

    e2_g = jnp.abs(e2_g_estimate)

    # floor guard: drop guided weight below `floor` x the largest score
    e2_g = jnp.where(e2_g >= floor * jnp.max(e2_g), e2_g, 0.0)

    # uniform guard: mix so that every vector keeps a strictly positive probability
    nchol = e2_g.shape[0]
    uniform = jnp.full((nchol,), 1.0 / nchol)
    total = jnp.sum(e2_g)
    guided = jnp.where(total > 0.0, e2_g / jnp.where(total > 0.0, total, 1.0), uniform)
    pi_g = (1.0 - uniform_mix) * guided + uniform_mix * uniform

    return pi_g


def _build_chol_sampling(pi_g, n_head):
    """Split the Cholesky vectors into an exact head and a sampled tail.

    The head is the `n_head` largest-probability vectors, summed exactly for every
    walker; the tail probabilities are renormalized over what is left.  Host-side,
    built once -- the split does not change during a run.

    Rule of thumb for n_head: a tail vector is drawn n_samples * pi_g times in
    expectation, so anything with pi_g > 1/n_samples is cheaper to put in the head
    than to keep re-drawing.  Check `1/sum(pi_g**2)` (the effective sample size):
    if it is much smaller than nchol the proposal is very peaked and the head
    should be larger.
    """
    pi_g = jnp.asarray(pi_g)
    order = jnp.argsort(-pi_g)
    head = jnp.sort(order[:n_head])
    tail = jnp.sort(order[n_head:])
    tail_prob = pi_g[tail]
    tail_prob = tail_prob / jnp.sum(tail_prob)
    return {"head": head, "tail": tail, "tail_prob": tail_prob}


@partial(jit, static_argnums=0)
def _calc_energy_pt(self, walker, ham_data, wave_data):
    # original function

    if self.mix_precision:
        rtype = jnp.float32
        ctype = jnp.complex64
    else:
        rtype = jnp.float64
        ctype = jnp.complex128
    
    nocc, norb = self.nelec[0], self.norb
    nchol_chunk = self.nchol_chunk  # nchol per chunk

    t2 = wave_data["t2"]
    h1 = ham_data["h1_bar"]
    chol = ham_data["chol_bar"]
    walker_bar = wave_data['exp_t1'] @ walker

    obar = jnp.linalg.det(walker_bar[:walker_bar.shape[1], :]) ** 2

    green = (walker_bar.dot(jnp.linalg.inv(walker_bar[: walker_bar.shape[1], :]))).T
    green_occ = green[:, nocc:]
    greenp = jnp.vstack((green_occ, -jnp.eye(norb - nocc)))
    rot_chol = chol[:, :nocc, :]
    nchol = chol.shape[0]
    # chunk_size = naux // nchol_chunk

    # 1 body energy
    hg = oe.contract("pi,pi->", h1[:nocc, :], green, backend="jax")
    e1_0 = 2 * hg

    t2g_c = oe.contract("iajb,ia->jb", t2, green[:nocc,nocc:], backend="jax")
    t2g_e = oe.contract("iajb,ib->ja", t2, green[:nocc,nocc:], backend="jax")
    t2_green_c = oe.contract("pb,jb,jq->pq", greenp, t2g_c, green, backend="jax")
    #(greenp @ t2g_c.T) @ green[:nocc,:]
    t2_green_e = oe.contract("pa,ja,jq->pq", greenp, t2g_e, green, backend="jax")
    #(greenp @ t2g_e.T) @ green[:nocc,:]
    t2_green = 2 * t2_green_c - t2_green_e
    t2g = 2 * t2g_c - t2g_e
    gt2g = oe.contract("ia,ia->", t2g, green[:nocc,nocc:], backend="jax")
    e1_2_1 = 2 * hg * gt2g
    e1_2_2 = -2 * oe.contract("ij,ij->", h1, t2_green, backend="jax")
    e1_2 = e1_2_1 + e1_2_2 # <exp(T1)HF|T2 h1|walker>/<exp(T1)HF|walker>

    # pad with zero cholesky vectors — contributes nothing to any contraction
    npad = (-nchol) % nchol_chunk
    chol = jnp.concatenate([chol, jnp.zeros((npad, norb, norb))], axis=0)
    rot_chol = jnp.concatenate([rot_chol, jnp.zeros((npad, nocc, norb))], axis=0)

    # reshape into chunks: (n_chunks, chunk_size, ...)
    nchunk = (nchol + npad) // nchol_chunk
    chol = chol.reshape(nchunk, nchol_chunk, norb, norb)
    rot_chol = rot_chol.reshape(nchunk, nchol_chunk, nocc, norb)

    # two body — scan over chunks, explicit contractions within a chunk
    def scan_chunk(carry, x):
        chol_c, rot_chol_c = x  # (chunk_size, norb, norb), (chunk_size, nocc, norb)

        gl = oe.contract("ir,gqr->giq", green, chol_c, backend="jax")
        gl_c = oe.contract("gii->g", gl[:, :, :nocc], backend="jax")
        e2_0_c = oe.contract("g,g->", gl_c, gl_c, backend="jax") * 2
        e2_0_e = -oe.contract("gij,gji->", gl[:, :, :nocc], gl[:, :, :nocc], backend="jax")
        carry[0] += e2_0_c + e2_0_e # NOTE: use this energy per g to build the esitimation!!!

        lt2g = oe.contract("gpr,pr->g", 
                            chol_c.astype(rtype), 
                            t2_green.astype(ctype), 
                            backend="jax")
        carry[1] += -oe.contract("g,g->", 
                                    lt2g.astype(ctype), 
                                    gl_c.astype(ctype), 
                                    backend="jax")

        lt2_green = oe.contract("gir,qr->giq", 
                                rot_chol_c.astype(rtype), 
                                t2_green.astype(ctype), 
                                backend="jax")
        # t_iajb |G_ia G_js Gp_pb| G_qr L_pr L_qs
        carry[2] += 0.5 * oe.contract("giq,giq->", 
                                        gl.astype(ctype), 
                                        lt2_green.astype(ctype), 
                                        backend="jax")

        # t_iajb G_ir G_js Gp_pa Gp_qb L_pr L_qs type
        glgp = oe.contract("gir,rb->gib", 
                            gl.astype(ctype), 
                            greenp.astype(ctype), 
                            backend="jax")
        lt2_c = oe.contract("gia,iajb->gjb", 
                            glgp.astype(ctype), 
                            t2.astype(rtype), 
                            backend="jax")
        lt2_e = oe.contract("gib,iajb->gja", 
                            glgp.astype(ctype), 
                            t2.astype(rtype), 
                            backend="jax")
        
        l2t2_c = oe.contract("gjb,gjb->", 
                                lt2_c.astype(ctype), 
                                glgp.astype(ctype), 
                                backend="jax").astype(jnp.complex128)
        l2t2_e = oe.contract("gja,gja->", 
                                lt2_e.astype(ctype), 
                                glgp.astype(ctype), 
                                backend="jax").astype(jnp.complex128)
        carry[3] += (2*l2t2_c - l2t2_e).astype(jnp.complex128)

        return carry, 0.0

    [e2_0, e2_2_2_1, e2_2_2_2, e2_2_3], _ = lax.scan(
        scan_chunk, [0.0, 0.0, 0.0, 0.0], (chol, rot_chol)
    )

    e2_2_1 = e2_0 * gt2g
    e2_2_2 = 4 * (e2_2_2_1 + e2_2_2_2)
    e2_2 = e2_2_1 + e2_2_2 + e2_2_3

    e0 = e1_0 + e2_0  # <psi|(h1+h2)|phi>/<psi|phi>
    e1 = e1_2 + e2_2  # <psi|t2(h1+h2)|phi>/<psi|phi>
    t2 = gt2g          # <psi|t2|phi>/<psi|phi>
    return obar, t2, e0, e1


@partial(jit, static_argnums=(0, 5))
def _calc_energy_pt_sto_chol(self, walker, ham_data, wave_data,
                             chol_sampling, n_samples, key):
    """Semistochastic version of _calc_energy_pt.

    Every accumulator in the scan of _calc_energy_pt is a plain sum over Cholesky
    vectors with no coupling between them, so the sum splits as

        sum_g  =  sum_{g in head}   +   sum_{g in tail}
                  (exact)               (importance sampled)

    The tail is estimated from `n_samples` draws g_m ~ tail_prob, each carrying the
    weight 1 / (n_samples * pi_{g_m}); the proposal cancels against the weight, so
    the estimate is unbiased for any positive tail_prob.  Error falls as
    1/sqrt(n_samples); the head costs nothing stochastically.

    Returns (obar, t2, e0, e1) exactly like _calc_energy_pt, so it is a drop-in.
    obar and t2 carry no Cholesky sum and remain exact; e0 and e1 are unbiased.
    """

    if self.mix_precision:
        rtype = jnp.float32
        ctype = jnp.complex64
    else:
        rtype = jnp.float64
        ctype = jnp.complex128

    nocc, norb = self.nelec[0], self.norb
    nchol_chunk = self.nchol_chunk

    t2 = wave_data["t2"]
    h1 = ham_data["h1_bar"]
    chol = ham_data["chol_bar"]
    walker_bar = wave_data['exp_t1'] @ walker

    obar = jnp.linalg.det(walker_bar[:walker_bar.shape[1], :]) ** 2

    green = (walker_bar.dot(jnp.linalg.inv(walker_bar[: walker_bar.shape[1], :]))).T
    green_occ = green[:, nocc:]
    greenp = jnp.vstack((green_occ, -jnp.eye(norb - nocc)))
    rot_chol = chol[:, :nocc, :]

    # ---------- Cholesky-independent pieces (identical to _calc_energy_pt) ----------
    hg = oe.contract("pi,pi->", h1[:nocc, :], green, backend="jax")
    e1_0 = 2 * hg

    t2g_c = oe.contract("iajb,ia->jb", t2, green[:nocc,nocc:], backend="jax")
    t2g_e = oe.contract("iajb,ib->ja", t2, green[:nocc,nocc:], backend="jax")
    t2_green_c = oe.contract("pb,jb,jq->pq", greenp, t2g_c, green, backend="jax")
    t2_green_e = oe.contract("pa,ja,jq->pq", greenp, t2g_e, green, backend="jax")
    t2_green = 2 * t2_green_c - t2_green_e
    t2g = 2 * t2g_c - t2g_e
    gt2g = oe.contract("ia,ia->", t2g, green[:nocc,nocc:], backend="jax")
    e1_2_1 = 2 * hg * gt2g
    e1_2_2 = -2 * oe.contract("ij,ij->", h1, t2_green, backend="jax")
    e1_2 = e1_2_1 + e1_2_2

    # ---------- weighted sum of the four per-Cholesky accumulators ----------
    def accumulate(indices, weights):
        """sum_m weights[m] * (per-vector terms of Cholesky vector indices[m])."""
        n = indices.shape[0]
        if n == 0:
            zero = jnp.zeros((), dtype=ctype)
            return zero, zero, zero, zero

        # pad to whole chunks; padded entries carry zero weight so contribute nothing
        npad = (-n) % nchol_chunk
        idx = jnp.concatenate([indices, jnp.zeros(npad, dtype=indices.dtype)])
        wts = jnp.concatenate([weights, jnp.zeros(npad, dtype=weights.dtype)])
        nchunk = (n + npad) // nchol_chunk
        idx = idx.reshape(nchunk, nchol_chunk)
        wts = wts.reshape(nchunk, nchol_chunk)

        def scan_chunk(carry, x):
            idx_c, w_c = x
            chol_c = chol[idx_c]              # (chunk, norb, norb)
            rot_chol_c = rot_chol[idx_c]      # (chunk, nocc, norb)
            w_c = w_c.astype(ctype)

            gl = oe.contract("ir,gqr->giq", green, chol_c, backend="jax")
            gl_occ = gl[:, :, :nocc]
            gl_c = oe.contract("gii->g", gl_occ, backend="jax")

            # -> e2_0
            a_g = 2 * oe.contract("g,g->g", gl_c, gl_c, backend="jax") \
                  - oe.contract("gij,gji->g", gl_occ, gl_occ, backend="jax")
            carry[0] += jnp.sum(w_c * a_g.astype(ctype))

            # -> e2_2_2_1
            lt2g = oe.contract("gpr,pr->g",
                               chol_c.astype(rtype),
                               t2_green.astype(ctype), backend="jax")
            b_g = -lt2g.astype(ctype) * gl_c.astype(ctype)
            carry[1] += jnp.sum(w_c * b_g)

            # -> e2_2_2_2
            lt2_green = oe.contract("gir,qr->giq",
                                    rot_chol_c.astype(rtype),
                                    t2_green.astype(ctype), backend="jax")
            c_g = 0.5 * oe.contract("giq,giq->g",
                                    gl.astype(ctype),
                                    lt2_green.astype(ctype), backend="jax")
            carry[2] += jnp.sum(w_c * c_g)

            # -> e2_2_3
            glgp = oe.contract("gir,rb->gib",
                               gl.astype(ctype),
                               greenp.astype(ctype), backend="jax")
            lt2_c = oe.contract("gia,iajb->gjb",
                                glgp.astype(ctype), t2.astype(rtype), backend="jax")
            lt2_e = oe.contract("gib,iajb->gja",
                                glgp.astype(ctype), t2.astype(rtype), backend="jax")
            l2t2_c = oe.contract("gjb,gjb->g",
                                 lt2_c.astype(ctype), glgp.astype(ctype), backend="jax")
            l2t2_e = oe.contract("gja,gja->g",
                                 lt2_e.astype(ctype), glgp.astype(ctype), backend="jax")
            d_g = (2 * l2t2_c - l2t2_e).astype(ctype)
            carry[3] += jnp.sum(w_c * d_g)

            return carry, 0.0

        zero = jnp.zeros((), dtype=ctype)
        out, _ = lax.scan(scan_chunk, [zero, zero, zero, zero], (idx, wts))
        return out[0], out[1], out[2], out[3]

    head = chol_sampling["head"]
    tail = chol_sampling["tail"]
    tail_prob = chol_sampling["tail_prob"]

    # head: exact, unit weight
    a_h, b_h, c_h, d_h = accumulate(head, jnp.ones(head.shape[0], dtype=ctype))

    # tail: n_samples draws with importance weight 1 / (n_samples * pi)
    if tail.shape[0] == 0:
        a_t = b_t = c_t = d_t = jnp.zeros((), dtype=ctype)
    else:
        sel = random.choice(key, tail.shape[0], shape=(n_samples,),
                            replace=True, p=tail_prob)
        samp_idx = tail[sel]
        samp_w = (1.0 / (n_samples * tail_prob[sel])).astype(ctype)
        a_t, b_t, c_t, d_t = accumulate(samp_idx, samp_w)

    e2_0_h     = a_h
    e2_2_2_1_h = b_h
    e2_2_2_2_h = c_h
    e2_2_3_h   = d_h

    e2_0_t     = a_t
    e2_2_2_1_t = b_t
    e2_2_2_2_t = c_t
    e2_2_3_t   = d_t

    # ---------- identical combination to _calc_energy_pt ----------
    e2_2_1_h = e2_0_h * gt2g
    e2_2_2_h = 4 * (e2_2_2_1_h + e2_2_2_2_h)
    e2_2_h = e2_2_1_h + e2_2_2_h + e2_2_3_h

    e2_2_1_t = e2_0_t * gt2g
    e2_2_2_t = 4 * (e2_2_2_1_t + e2_2_2_2_t)
    e2_2_t = e2_2_1_t + e2_2_2_t + e2_2_3_t

    t2 = gt2g
    e0_h = e1_0 + e2_0_h
    e1_h = e1_2 + e2_2_h
    e0_t = e2_0_t
    e1_t = e2_2_t
    return obar, t2, e0_h, e1_h, e0_t, e1_t


In [41]:
# ---- validation: head-only must reproduce the exact contraction, and the ----
# ---- sampled estimator must be unbiased with 1/sqrt(M) error ----
import numpy as np

walker = prop_data["walkers"][0]
ref = _calc_energy_pt(trial, walker, ham_data, wave_data)
E_ref = complex(h0 + ref[2] + ref[3] - ref[1] * ref[2])
print(f"exact  E = {E_ref:.10f}")

# guide from the HF determinant (build it once, reuse for the whole run)
pi_g = _prop_chol(trial, prop_data["walkers"][0], ham_data, wave_data)
nchol = pi_g.shape[0]
print(f"pi_g: min={float(jnp.min(pi_g)):.2e} max={float(jnp.max(pi_g)):.2e} "
      f"uniform={1/nchol:.2e}  ESS={1/float(jnp.sum(pi_g**2)):.1f}/{nchol}")

# (1) whole set in the head -> tail empty -> must equal _calc_energy_pt
full = _build_chol_sampling(pi_g, nchol)
full_out = _calc_energy_pt_sto_chol(trial, walker, ham_data, wave_data, full, 8, random.PRNGKey(0))
print(ref)
print(full_out)

exact  E = -1.1307473282-0.0000339405j
pi_g: min=2.17e-04 max=7.53e-01 uniform=2.17e-02  ESS=1.6/46
(Array(0.34092338+0.82581932j, dtype=complex128), Array(0.00175075+0.00085472j, dtype=complex128), Array(-1.59541406+0.00219784j, dtype=complex128), Array(-0.03812831-0.00359156j, dtype=complex128))
(Array(0.34092338+0.82581932j, dtype=complex128), Array(0.00175075+0.00085472j, dtype=complex128), Array(-1.59541406+0.00219784j, dtype=complex128), Array(-0.03812831-0.00359156j, dtype=complex128), Array(0.+0.j, dtype=complex128), Array(0.+0.j, dtype=complex128))


In [43]:
samp = _build_chol_sampling(pi_g, max(1, sum(pi_g > 0.001)))
out = _calc_energy_pt_sto_chol(trial, walker, ham_data, wave_data, samp, 8, random.PRNGKey(0))
print(ref)
print(out)

(Array(0.34092338+0.82581932j, dtype=complex128), Array(0.00175075+0.00085472j, dtype=complex128), Array(-1.59541406+0.00219784j, dtype=complex128), Array(-0.03812831-0.00359156j, dtype=complex128))
(Array(0.34092338+0.82581932j, dtype=complex128), Array(0.00175075+0.00085472j, dtype=complex128), Array(-1.59549736+0.00235835j, dtype=complex128), Array(-0.0358487-0.00377352j, dtype=complex128), Array(-0.00070297-0.00014446j, dtype=complex128), Array(-0.00333993+0.00025942j, dtype=complex128))


In [36]:
sum(pi_g > 0.001)

Array(13, dtype=int64, weak_type=True)

In [53]:
# (2) unbiasedness and error scaling
samp = _build_chol_sampling(pi_g, max(1, sum(pi_g > 0.01)))
print(f"head={int(samp['head'].shape[0])}  tail={int(samp['tail'].shape[0])}")
R = 2000
print(f"{'M':>6} {'mean E':>18} {'bias':>12} {'|bias|/SEM':>11} {'std':>12}")
for M in (8, 32, 128):
    keys = random.split(random.PRNGKey(2024), R)
    o = jax.vmap(lambda k: _calc_energy_pt_sto_chol(
        trial, walker, ham_data, wave_data, samp, M, k))(keys)
    e0 = (o[2] + o[4])
    e1 = (o[3] + o[5])
    E_s = np.asarray(h0 + e0 + e1 - o[1] * e0).real
    m, sd = E_s.mean(), E_s.std(ddof=1)
    print(f"{M:6d} {m:18.10f} {m - E_ref.real:+12.2e} "
          f"{abs(m - E_ref.real) / (sd / np.sqrt(R)):11.2f} {sd:12.4e}")

head=3  tail=43
     M             mean E         bias  |bias|/SEM          std
     8      -1.1307176472    +2.97e-05        0.13   1.0614e-02
    32      -1.1307751635    -2.78e-05        0.23   5.3276e-03
   128      -1.1307104528    +3.69e-05        0.63   2.6173e-03


In [ ]:
# ---------------------------------------------------------------------------
# In-place variant.  Two specialised scans over the Cholesky index:
#
#   scan_chol_chunk_e2_0 : ALL gamma, exact.  Needs only gl = green.chol, no T2.
#                          Gives both the exact e2_0 and the per-gamma e2_0_g that
#                          _prop_chol_in_place turns into the proposal.  We have to
#                          evaluate this anyway, so e2_0 is never sampled.
#
#   scan_chol_chunk_e2_2 : only the three accumulators that contract with T2
#                          (e2_2_2_1, e2_2_2_2, e2_2_3).  Head exactly, tail from
#                          n_samples importance-weighted draws.  The e2_0 exchange
#                          trace "gij,gji->g" is never formed here -- gl_c is built
#                          only because e2_2_2_1 needs it.
#
# e2_2_1 = e2_0 * gt2g is exact too, since both factors are.  So the ONLY sampled
# quantity is 4*(e2_2_2_1 + e2_2_2_2) + e2_2_3, and e0 is returned exact:
# e0_t is identically zero, kept so the 6-value return matches
# _calc_energy_pt_sto_chol.
#
# Why this is the right split: the T2 contractions ("gia,iajb->gjb" and friends)
# scale as nocc^2 nvir^2 per Cholesky vector and dominate everything else, while
# e2_0_g costs only the gl build.  So we pay the cheap part in full and sample the
# expensive part.  On H2/ccpVDZ (nocc=1) that trade is poor; with a real virtual
# space it is the whole cost.
#
# Unbiasedness is unaffected by the proposal adapting to the walker: pi_g is a
# deterministic function of the walker, fixed before any draw.  Caveat: |e2_0_g| is
# a *surrogate* score -- it ranks vectors by a quantity that is no longer the one
# being sampled.  The variance-optimal score would be |4(b_g + c_g) + d_g|, which
# costs exactly what we are avoiding.
#
# n_head and n_samples are static (they set array shapes): changing them recompiles.
# ---------------------------------------------------------------------------

@partial(jit, static_argnums=(0, 4, 5))
def _calc_energy_pt_sto_chol_in_place(self, walker, ham_data, wave_data,
                                      n_head, n_samples, key,
                                      floor=1.0e-6, uniform_mix=0.01):
    if self.mix_precision:
        rtype, ctype = jnp.float32, jnp.complex64
    else:
        rtype, ctype = jnp.float64, jnp.complex128

    nocc, norb = self.nelec[0], self.norb
    nchol_chunk = self.nchol_chunk

    t2 = wave_data["t2"]
    h1 = ham_data["h1_bar"]
    chol = ham_data["chol_bar"]
    nchol = chol.shape[0]
    walker_bar = wave_data["exp_t1"] @ walker

    obar = jnp.linalg.det(walker_bar[: walker_bar.shape[1], :]) ** 2
    green = (walker_bar.dot(jnp.linalg.inv(walker_bar[: walker_bar.shape[1], :]))).T
    green_occ = green[:, nocc:]
    greenp = jnp.vstack((green_occ, -jnp.eye(norb - nocc)))
    rot_chol = chol[:, :nocc, :]

    hg = oe.contract("pi,pi->", h1[:nocc, :], green, backend="jax")
    e1_0 = 2 * hg
    t2g_c = oe.contract("iajb,ia->jb", t2, green[:nocc, nocc:], backend="jax")
    t2g_e = oe.contract("iajb,ib->ja", t2, green[:nocc, nocc:], backend="jax")
    t2_green_c = oe.contract("pb,jb,jq->pq", greenp, t2g_c, green, backend="jax")
    t2_green_e = oe.contract("pa,ja,jq->pq", greenp, t2g_e, green, backend="jax")
    t2_green = 2 * t2_green_c - t2_green_e
    t2g = 2 * t2g_c - t2g_e
    gt2g = oe.contract("ia,ia->", t2g, green[:nocc, nocc:], backend="jax")
    e1_2 = 2 * hg * gt2g - 2 * oe.contract("ij,ij->", h1, t2_green, backend="jax")

    # ================= pass 1: e2_0, exact, every gamma, no T2 =================
    def scan_chol_chunk_e2_0(carry, x):
        """e2_0_g for one chunk.  Only needs gl = green.chol -- no T2 anywhere."""
        idx_c, keep_c = x
        gl = oe.contract("ir,gqr->giq", green, chol[idx_c], backend="jax")
        gl_occ = gl[:, :, :nocc]
        gl_c = oe.contract("gii->g", gl_occ, backend="jax")
        e2_0_g = (2 * oe.contract("g,g->g", gl_c, gl_c, backend="jax")
                  - oe.contract("gij,gji->g", gl_occ, gl_occ, backend="jax"))
        e2_0_g = e2_0_g.astype(ctype) * keep_c.astype(ctype)
        return carry + jnp.sum(e2_0_g), e2_0_g

    npad = (-nchol) % nchol_chunk
    nchunk = (nchol + npad) // nchol_chunk
    idx_all = jnp.concatenate([jnp.arange(nchol, dtype=jnp.int32),
                               jnp.zeros(npad, dtype=jnp.int32)]).reshape(nchunk, nchol_chunk)
    keep = jnp.concatenate([jnp.ones(nchol), jnp.zeros(npad)]).reshape(nchunk, nchol_chunk)
    e2_0, e2_0_chunks = lax.scan(scan_chol_chunk_e2_0,
                                 jnp.zeros((), dtype=ctype), (idx_all, keep))
    e2_0_g = e2_0_chunks.reshape(-1)[:nchol]

    # ---- proposal from e2_0_g, head/tail split ----
    pi_g = _prop_chol_in_place(self, e2_0_g, floor, uniform_mix)
    order = jnp.argsort(-pi_g)
    head = jnp.sort(order[:n_head])
    tail = jnp.sort(order[n_head:])
    tail_prob = pi_g[tail]
    tail_prob = tail_prob / jnp.sum(tail_prob)

    # =========== pass 2: only the e2_2 terms that contract with T2 ===========
    def scan_chol_chunk_e2_2(carry, x):
        """The three T2-contracted accumulators for one chunk, weighted.

        No e2_0 work here: the 'gij,gji->g' exchange trace is never formed, and
        gl_c is built only because e2_2_2_1 needs it.
        """
        idx_c, w_c = x
        chol_c, rot_chol_c = chol[idx_c], rot_chol[idx_c]
        w_c = w_c.astype(ctype)

        gl = oe.contract("ir,gqr->giq", green, chol_c, backend="jax")
        gl_c = oe.contract("gii->g", gl[:, :, :nocc], backend="jax")

        # e2_2_2_1
        lt2g = oe.contract("gpr,pr->g", chol_c.astype(rtype),
                           t2_green.astype(ctype), backend="jax")
        carry[0] += jnp.sum(w_c * (-lt2g.astype(ctype) * gl_c.astype(ctype)))

        # e2_2_2_2
        lt2_green = oe.contract("gir,qr->giq", rot_chol_c.astype(rtype),
                                t2_green.astype(ctype), backend="jax")
        carry[1] += jnp.sum(w_c * 0.5 * oe.contract("giq,giq->g", gl.astype(ctype),
                            lt2_green.astype(ctype), backend="jax"))

        # e2_2_3
        glgp = oe.contract("gir,rb->gib", gl.astype(ctype),
                           greenp.astype(ctype), backend="jax")
        lt2_c = oe.contract("gia,iajb->gjb", glgp.astype(ctype), t2.astype(rtype), backend="jax")
        lt2_e = oe.contract("gib,iajb->gja", glgp.astype(ctype), t2.astype(rtype), backend="jax")
        l2t2_c = oe.contract("gjb,gjb->g", lt2_c.astype(ctype), glgp.astype(ctype), backend="jax")
        l2t2_e = oe.contract("gja,gja->g", lt2_e.astype(ctype), glgp.astype(ctype), backend="jax")
        carry[2] += jnp.sum(w_c * (2 * l2t2_c - l2t2_e).astype(ctype))
        return carry, 0.0

    def accumulate_e2_2(indices, weights):
        n = indices.shape[0]
        z = jnp.zeros((), dtype=ctype)
        if n == 0:
            return z, z, z
        npad2 = (-n) % nchol_chunk
        idx = jnp.concatenate([indices, jnp.zeros(npad2, dtype=indices.dtype)])
        wts = jnp.concatenate([weights, jnp.zeros(npad2, dtype=weights.dtype)])
        nch2 = (n + npad2) // nchol_chunk
        out, _ = lax.scan(scan_chol_chunk_e2_2, [z, z, z],
                          (idx.reshape(nch2, nchol_chunk), wts.reshape(nch2, nchol_chunk)))
        return out[0], out[1], out[2]

    # head: exact
    b_h, c_h, d_h = accumulate_e2_2(head, jnp.ones(head.shape[0], dtype=ctype))
    # tail: sampled
    if tail.shape[0] == 0:
        b_t = c_t = d_t = jnp.zeros((), dtype=ctype)
    else:
        sel = random.choice(key, tail.shape[0], shape=(n_samples,), replace=True, p=tail_prob)
        samp_w = (1.0 / (n_samples * tail_prob[sel])).astype(ctype)
        b_t, c_t, d_t = accumulate_e2_2(tail[sel], samp_w)

    # e2_2_1 = e2_0 * gt2g is exact, since e2_0 is exact
    e2_2_h = e2_0 * gt2g + 4 * (b_h + c_h) + d_h
    e2_2_t = 4 * (b_t + c_t) + d_t

    e0 = e1_0 + e2_0          # fully exact
    e1_h = e1_2 + e2_2_h
    e1_t = e2_2_t
    e1 = e1_h + e1_t
    return obar, gt2g, e0, e1


In [58]:
# ---- validation of the in-place variant ----
import numpy as np

def _energy(o):
    e0 = o[2] + o[4]
    e1 = o[3] + o[5]
    return h0 + e0 + e1 - o[1] * e0

walker = prop_data["walkers"][0]
ref = _calc_energy_pt(trial, walker, ham_data, wave_data)
E_ref = complex(h0 + ref[2] + ref[3] - ref[1] * ref[2])
nchol = ham_data["chol_bar"].shape[0]
print(f"exact E = {E_ref:.10f}")

# (1) head = all gamma -> tail empty -> must reproduce the exact contraction,
#     and e0 must be exact for any head size since e2_0 is never sampled
o = _calc_energy_pt_sto_chol_in_place(trial, walker, ham_data, wave_data,
                                      nchol, 8, random.PRNGKey(0))
print(f"head=all : |dE| = {abs(complex(_energy(o)) - E_ref):.2e}")
o = _calc_energy_pt_sto_chol_in_place(trial, walker, ham_data, wave_data,
                                      max(1, round(nchol / 8)), 8, random.PRNGKey(0))
print(f"small head: |de0| = {abs(complex(o[2] + o[4]) - complex(ref[2])):.2e}"
      f"   e0_t = {complex(o[4]):.1e}   (e0 exact, e0_t identically zero)")

# (2) unbiasedness and variance vs the fixed-HF-guide version
pi_ext = _prop_chol(trial, prop_data["walkers"][0], ham_data, wave_data)
NH = max(1, round(nchol / 8))
samp_ext = _build_chol_sampling(pi_ext, NH)
R = 2000
print(f"\nhead={NH}, {R} draws")
print(f"{'M':>5}  {'variant':>22} {'mean E':>17} {'bias':>11} {'|b|/SEM':>8} {'std':>11}")
for M in (8, 32):
    keys = random.split(random.PRNGKey(2024), R)
    runs = [
        ("external HF proposal", jax.vmap(lambda k: _calc_energy_pt_sto_chol(
            trial, walker, ham_data, wave_data, samp_ext, M, k))(keys)),
        ("in-place, e2_0 exact", jax.vmap(lambda k: _calc_energy_pt_sto_chol_in_place(
            trial, walker, ham_data, wave_data, NH, M, k))(keys)),
    ]
    for name, o in runs:
        E = np.asarray(_energy(o)).real
        m, sd = E.mean(), E.std(ddof=1)
        print(f"{M:5d}  {name:>22} {m:17.10f} {m - E_ref.real:+11.2e} "
              f"{abs(m - E_ref.real) / (sd / np.sqrt(R)):8.2f} {sd:11.4e}")


exact E = -1.1307473282-0.0000339405j
head=all : |dE| = 1.73e-18
small head: |de0| = 0.00e+00   e0_t = 0.0e+00+0.0e+00j   (e0 exact, e0_t identically zero)

head=6, 2000 draws
    M                 variant            mean E        bias  |b|/SEM         std
    8    external HF proposal     -1.1305372866   +2.10e-04     1.32  7.1130e-03
    8    in-place, e2_0 exact     -1.1306083195   +1.39e-04     1.15  5.3938e-03
   32    external HF proposal     -1.1307592229   -1.19e-05     0.15  3.6076e-03
   32    in-place, e2_0 exact     -1.1307625821   -1.53e-05     0.24  2.7899e-03
